Az Ollama fusson a gépeden (ha még nincs meg a modell: `ollama pull gemma4:e4b`). Futtasd sorban: **Shift + Enter**. Ha valami elakad: **Restart**, aztán **Run All**.

In [ ]:
import json
import re
import time
import urllib.request

import pandas as pd

import kurzus
from kurzus import chat, kerdez, utolso_hasznalat
from kurzus import ellenoriz, ellenoriz_feltetel, ellenoriz_fuggveny

## 1. Miért kódból, és mi az az API?

*Ezt az óra elején, a diákon beszéljük meg.*

## 2. Az első hívás, kézzel

A kérés egy Python-szótár: melyik modell válaszoljon, és az üzenetek listája.

In [ ]:
keres_adat = {
    "model": "gemma4:e4b",
    "messages": [
        {"role": "user", "content": "Egy mondatban: mit mér a fogyasztóiár-index?"},
    ],
    "temperature": 0,
}

# Ez a szöveg megy át a hálózaton: a szótár JSON-ként.
print(json.dumps(keres_adat, ensure_ascii=False))

Elküldjük a helyi Ollamának (kb. 20 másodperc). Ha `Connection refused` a hiba, nem fut az Ollama.

In [ ]:
cim = "http://localhost:11434/v1/chat/completions"

keres = urllib.request.Request(
    cim,
    data=json.dumps(keres_adat).encode("utf-8"),
    headers={"Content-Type": "application/json", "Authorization": "Bearer ollama"},
)
with urllib.request.urlopen(keres) as http_valasz:
    valasz = json.loads(http_valasz.read())

print(json.dumps(valasz, ensure_ascii=False, indent=2))

### Feladat 2.1 – olvasás

Hány tokenből állt a kérdésed (`prompt_tokens`)? Olvasd ki a fenti kiírásból, és írd be a `None` helyére.

In [ ]:
tipp_2_1 = None  # a None helyére írd a fenti kiírásból kiolvasott számot

In [ ]:
ellenoriz("tipp_2_1", vart=valasz["usage"]["prompt_tokens"])

### Feladat 2.2 – Copilottal

Kérd meg a Copilotot (**Ctrl + I**, Mac: **⌘ + I**):

> Vedd ki a `valasz` szótárból a modell válaszának szövegét (choices → első elem → message → content), és tedd a `valasz_szoveg` változóba. Írd ki.

**Olvasd el, mielőtt elfogadod:** hány szögletes zárójel van a kódban, és mindegyik mit választ ki?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz("valasz_szoveg", vart=valasz["choices"][0]["message"]["content"])

Ugyanez függvényként, a későbbi részekhez:

In [ ]:
def kuld(keres_adat):
    """Elküldi a kérést a modellnek, és a teljes választ szótárként adja vissza."""
    keres = urllib.request.Request(
        cim,
        data=json.dumps(keres_adat).encode("utf-8"),
        headers={"Content-Type": "application/json", "Authorization": "Bearer ollama"},
    )
    with urllib.request.urlopen(keres) as http_valasz:
        return json.loads(http_valasz.read())

## 3. Ugyanez a `kurzus.py`-ban

Ez a függvény küldi el a kérést:

In [ ]:
import inspect

print(inspect.getsource(kurzus._keres))

A `chat` és a `kerdez`:

In [ ]:
print(inspect.getsource(kurzus.chat))
print(inspect.getsource(kurzus.kerdez))

### Feladat 3.1 – olvasás

Olvasd el a `kerdez` függvényt. Ha meghívod így: `kerdez("Mi az infláció?", rendszer="Válaszolj röviden.")`, **hány üzenet** lesz a listában, amit elküld a modellnek?

In [ ]:
tipp_3_1 = None  # a None helyére írd a tipped

In [ ]:
# Ellenőrzés: egy pillanatra kicseréljük a kurzus.chat függvényt egy "kémre",
# amely nem küld el semmit, csak feljegyzi, milyen üzenetlistát kapott.
feljegyzett = []


def kem(uzenetek, **tobbi):
    feljegyzett.append(uzenetek)
    return {"content": ""}


eredeti_chat = kurzus.chat
kurzus.chat = kem
kurzus.kerdez("Mi az infláció?", rendszer="Válaszolj röviden.")
kurzus.chat = eredeti_chat     # visszatesszük az igazit

print(feljegyzett[0])
ellenoriz("tipp_3_1", vart=len(feljegyzett[0]))

Mostantól így kérdezünk (gondolkodás nélkül, pár másodperc):

In [ ]:
print(kerdez("Egy mondatban: mit mér a fogyasztóiár-index?", gondolkodas="none"))
print(utolso_hasznalat)

## 4. Szolgáltató és API-kulcs

A minta beállítófájl:

In [ ]:
from pathlib import Path

# A .env.pelda a kurzus.py mellett van.
pelda = Path(kurzus.__file__).resolve().parent / ".env.pelda"
print(pelda.read_text(encoding="utf-8") if pelda.exists() else "Nincs meg a .env.pelda: töltsd le újra a notebookok.zip-et.")

### Feladat 4.1 – hibakeresés

Ez a kód lefut. Mi lesz belőle, amint a repó felkerül a GitHubra?

In [ ]:
# Egy hallgató kódja. A kulcs kitalált, de a formája olyan, mint egy igazié.
API_KULCS = "sk-proj-Xq7kitalalt1234567890abcdef"
fejlec = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KULCS}"}

Javítsd ki: a kulcs a `kurzus.beallitas("LLM_API_KEY")` függvényből jöjjön, és a `kulcs_javitott` változóba kerüljön.

In [ ]:
# Ide kerül a javítás

In [ ]:
ellenoriz_feltetel("kulcs_javitott", lambda kulcs: kulcs == kurzus.beallitas("LLM_API_KEY"),
                   "a kulcs a beállításokból (.env) jön, nem a kódból")

Végül írd be ezt a sort a mappa `AGENTS.md` fájljába, a *Biztonság* alá, hogy a Copilot is tudja:

> - API-kulcsot soha ne írj a kódba. A kulcs a `.env` fájlban van, azt ne nyisd meg és ne commitold.

## 5. Szerepek és a rendszerprompt

Ugyanaz a kérdés, három rendszerprompttal (gondolkodás nélkül):

In [ ]:
KERDES = "Mekkora a Magyar Nemzeti Bank inflációs célja?"

rendszerpromptok = {
    "nincs": None,
    "rövid": "Válaszolj röviden, magyarul, közgazdász hallgatóknak. Legfeljebb két mondat.",
    "csak szám": "Csak egyetlen számot írj, mértékegység és magyarázat nélkül.",
}

kiserlet = {}
for nev, rendszer in rendszerpromptok.items():
    kiserlet[nev] = kerdez(KERDES, rendszer=rendszer, gondolkodas="none")
    print(f"--- rendszerprompt: {nev}  ({utolso_hasznalat['prompt_tokens']} bemeneti token)")
    print(kiserlet[nev])
    print()

Előre megírt feltételek a válaszokhoz:

In [ ]:
def van_benne_szam(szoveg):
    """Van-e a szövegben legalább egy számjegy."""
    return re.search(r"\d", szoveg) is not None


def csak_szam(szoveg):
    """A szöveg egyetlen szám-e (pl. "3" vagy "2,5"), semmi más."""
    return re.fullmatch(r"\s*-?\d+([.,]\d+)?\s*", szoveg) is not None


for nev, szoveg in kiserlet.items():
    print(f"{nev:10} hossz: {len(szoveg):4} karakter | van benne szám: {van_benne_szam(szoveg)} | csak szám: {csak_szam(szoveg)}")

### Feladat 5.1 – Copilottal

Írass a Copilottal egy negyedik változatot, egy saját rendszerprompttal:

> Kérdezd meg a `KERDES`-t a `kerdez` függvénnyel, ezzel a rendszerprompttal: „Válaszolj egyetlen magyar mondatban, és a mondat végén zárójelben írd meg, honnan származik az adat.” Használd a `gondolkodas="none"` beállítást. A választ tedd a `valasz_forrassal` változóba, és írd ki.

Az ellenőrzés két előre megírt feltételt néz: van-e a válaszban szám, és van-e benne zárójel.

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz_feltetel("valasz_forrassal", lambda v: van_benne_szam(v) and "(" in v and ")" in v,
                   "van benne szám, és zárójelben a forrás")

### És igaz is?

Ez a cella kikeresi az MNB inflációs célját két Inflációs jelentésből (az első futáskor letölti őket).

In [ ]:
import pymupdf  # PDF-olvasó csomag

forrasok = pd.read_csv("data/mnb/forrasok.csv", sep=";")
for jelentes in ["2024-09", "2026-09"]:
    sor = forrasok[forrasok["jelentes"] == jelentes].iloc[0]
    pdf = pymupdf.open(kurzus.letolt(sor["url"], "data/mnb/" + sor["fajl"]))
    for oldalszam, oldal in enumerate(pdf, start=1):
        for szovegsor in oldal.get_text().splitlines():
            if "középtávú cél" in szovegsor:
                print(f"MNB Inflációs jelentés {jelentes}, PDF {oldalszam}. oldal:\n  {szovegsor.strip()}")

print("\nA modell válasza (csak szám):", kiserlet["csak szám"])

## 6. A motor belseje

### Feladat 6.1 – olvasás

Egy játék-„modell”: minden szóhoz megmondja, mi jön utána. Olvasd el a ciklust, és tippeld meg: **hány elem lesz a `szavak` listában** a végén?

In [ ]:
tipp_6_1 = None  # a None helyére írd a tipped

In [ ]:
# Olvasd el, és csak a tipped beírása után futtasd!
kovetkezo = {
    "<kezdet>": "Az",
    "Az": "infláció",
    "infláció": "2024-ben",
    "2024-ben": "mérséklődött",
    "mérséklődött": ".",
    ".": "<stop>",
}

szavak = []
szo = "<kezdet>"
while True:
    szo = kovetkezo[szo]
    if szo == "<stop>":
        break
    szavak.append(szo)

print(" ".join(szavak))
ellenoriz("tipp_6_1", vart=len(szavak))

Ha elfogy a megengedett hossz (`max_tokens`):

In [ ]:
rovid_keres = {
    "model": "gemma4:e4b",
    "messages": [{"role": "user", "content": "Mi az infláció? Két mondatban."}],
    "max_tokens": 8,               # legfeljebb 8 kimeneti token
    "reasoning_effort": "none",    # gondolkodás nélkül
}
eredmeny = kuld(rovid_keres)
print(eredmeny["choices"][0]["message"]["content"])
print("finish_reason:", eredmeny["choices"][0]["finish_reason"])

### Gondolkodás a válasz előtt

Ugyanez 8 tokennel, de bekapcsolt gondolkodással:

In [ ]:
rovid_keres.pop("reasoning_effort")    # most nem kapcsoljuk ki a gondolkodást
eredmeny = kuld(rovid_keres)
uzenet_8 = eredmeny["choices"][0]["message"]
print("gondolkodás:", repr(uzenet_8.get("reasoning", "")))
print("válasz:     ", repr(uzenet_8["content"]))

A 2. rész válaszának gondolkodása és válasza:

In [ ]:
uzenet = valasz["choices"][0]["message"]
gondolkodas_szoveg = uzenet.get("reasoning", "")   # felhőszolgáltatónál ez a mező hiányozhat

print("A gondolkodás eleje:", gondolkodas_szoveg[:300], "...")
print()
print("A válasz:", uzenet["content"])

### Feladat 6.2 – Copilottal

> Számold ki, hányszor hosszabb (karakterben) a `gondolkodas_szoveg`, mint a modell válasza (`uzenet["content"]`). Az eredmény a `hanyszor_hosszabb` változóba kerüljön.

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz("hanyszor_hosszabb", vart=len(gondolkodas_szoveg) / len(uzenet["content"]))

Ugyanaz a kérdés gondolkodással és nélküle (kb. fél perc):

In [ ]:
kerdes_ar = ("Egy termék ára 2023-ban 1000 Ft volt, 2024-ben 10%-kal, 2025-ben további 10%-kal drágult. "
             "Mennyibe kerül 2025-ben? Egy rövid mondatban válaszolj.")

for mod in [None, "none"]:
    kezdes = time.perf_counter()
    valasz_ar = chat([{"role": "user", "content": kerdes_ar}], gondolkodas=mod)
    ido = time.perf_counter() - kezdes
    cimke = "gondolkodással " if mod is None else "gondolkodás nélkül"
    print(f"{cimke}: {ido:5.1f} mp, {utolso_hasznalat['completion_tokens']:4} kimeneti token | {valasz_ar['content']}")

print("Python:", round(1000 * 1.10 * 1.10), "Ft")

## 7. A modellnek nincs memóriája

Írd át a nevet a sajátodra, és futtasd le a két cellát.

In [ ]:
nevem = "Bálint"   # írd át a sajátodra

elso_kerdes = {"role": "user", "content": f"Szia! {nevem} vagyok, közgazdász hallgató."}
elso_valasz = chat([elso_kerdes], gondolkodas="none")
print(elso_valasz["content"])

In [ ]:
# Egy új kérés, előzmények nélkül:
valasz_elozmeny_nelkul = kerdez("Mi az én nevem?", gondolkodas="none")
print(valasz_elozmeny_nelkul)

# A magyar ragozás miatt (Anna → Annának) a név utolsó magánhangzóját levágjuk a kereséshez.
nev_toredek = nevem.lower().rstrip("ae")
print("\nBenne van a neved a válaszban?", nev_toredek in valasz_elozmeny_nelkul.lower())

### Feladat 7.1 – Copilottal

> Készíts egy `uzenetek` listát, amelyben sorban benne van: az `elso_kerdes`, a modell első válasza (`elso_valasz`), és egy új felhasználói üzenet: „Mi az én nevem?”. Küldd el a `chat` függvénnyel, `gondolkodas="none"` beállítással, és a válasz szövegét (`content`) tedd a `valasz_elozmennyel` változóba. Írd ki.

**Olvasd el, mielőtt elfogadod:** milyen szerepe (`role`) van a lista második elemének?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz_feltetel("valasz_elozmennyel", lambda v: nev_toredek in v.lower(),
                   "a válaszban szerepel a neved, mert az előzményt is elküldted")

## 8. Tokenek és költség

Ugyanaz a mondat angolul és magyarul: hány token?

In [ ]:
mondatok = {
    "angol": "The central bank raised the base rate because inflation was high.",
    "magyar": "A jegybank megemelte az alapkamatot, mert magas volt az infláció.",
    "üres": "",
}
for nyelv, mondat in mondatok.items():
    meres = kuld({"model": "gemma4:e4b", "max_tokens": 1, "reasoning_effort": "none",
                  "messages": [{"role": "user", "content": mondat}]})
    print(f"{nyelv:7} {len(mondat):3} karakter → {meres['usage']['prompt_tokens']:3} bemeneti token")

Kitalált, kerek példaárak:

In [ ]:
AR_BE = 0.50   # USD / 1 millió bemeneti token, kitalált példaár
AR_KI = 2.00   # USD / 1 millió kimeneti token, kitalált példaár

### Feladat 8.1 – Copilottal

> Írj egy `koltseg(prompt_tokens, completion_tokens, ar_be, ar_ki)` függvényt, amely visszaadja egy hívás költségét dollárban. Az `ar_be` és az `ar_ki` az 1 millió tokenre vetített ár.

Olvasd el: elosztja valahol egymillióval? És a bemeneti tokeneket a bemeneti árral szorozza?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
def _koltseg(be, ki, ar_be, ar_ki):
    return (be * ar_be + ki * ar_ki) / 1_000_000


ellenoriz_fuggveny("koltseg", [((1_000_000, 0, AR_BE, AR_KI), _koltseg(1_000_000, 0, AR_BE, AR_KI)),
                               ((0, 1_000_000, AR_BE, AR_KI), _koltseg(0, 1_000_000, AR_BE, AR_KI)),
                               ((30, 270, AR_BE, AR_KI), _koltseg(30, 270, AR_BE, AR_KI))],
                   tures=1e-9)

### Feladat 8.2 – Copilottal

> A `koltseg` függvénnyel és a `valasz["usage"]` tokenszámaival számold ki, mennyibe kerülne, ha 30 hallgató egyenként 50 ilyen hívást küldene, `AR_BE` és `AR_KI` áron. Az eredmény az `osztaly_koltseg` változóba kerüljön, dollárban.

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
_h = valasz["usage"]
ellenoriz("osztaly_koltseg", vart=30 * 50 * _koltseg(_h["prompt_tokens"], _h["completion_tokens"], AR_BE, AR_KI),
          tures=1e-6)

## 9. Amikor a modell magabiztosan téved

A teljes HICP-táblázat a kérdésben, gondolkodással (kb. fél perc):

In [ ]:
adat = pd.read_csv("data/hicp_osszesen.csv", sep=";")
tablazat = adat.to_csv(sep=";", index=False)

kerdes_atlag = (
    "Itt a magyar havi infláció (HICP, éves változás, %), az Eurostat adatai:\n\n"
    + tablazat
    + "\nMennyi volt a havi értékek átlaga 2023 januárja és 2024 decembere között? "
    + "Csak a számot írd le, két tizedesre."
)
modell_valasza = kerdez(kerdes_atlag)
print("A modell válasza:", modell_valasza)
print("Tokenek:", utolso_hasznalat)

### Feladat 9.1 – Copilottal

> Számold ki az `adat` táblából a 2023-as és 2024-es havi `hicp_ev_ev` értékek átlagát (azokat a sorokat, ahol a `honap` 2023-mal vagy 2024-gyel kezdődik). Az eredmény az `atlag_2023_2024` változóba kerüljön.

Olvasd el: pontosan ezt a két évet választja ki? Hány sor marad a szűrés után (hány hónap van két évben)?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
# A helyes érték a nyers adatból, a modelltől függetlenül.
helyes_atlag = adat[adat["honap"].str[:4].isin(["2023", "2024"])]["hicp_ev_ev"].mean()
ellenoriz("atlag_2023_2024", vart=helyes_atlag)

A modell száma és a Python száma:

In [ ]:
szamok = re.findall(r"-?\d+(?:[.,]\d+)?", modell_valasza)
modell_szama = float(szamok[-1].replace(",", ".")) if szamok else None

print(f"A modell szerint: {modell_szama}")
print(f"A Python szerint: {helyes_atlag:.2f}")
if modell_szama is None:
    print("A modell válaszában nem volt szám.")
elif abs(modell_szama - helyes_atlag) <= 0.01:
    print("Most eltalálta. Kérdezz rá egy másik időszakra (például a teljes táblázatra): nem mindig sikerül.")
else:
    print(f"Eltérés: {abs(modell_szama - helyes_atlag):.2f} százalékpont. A modell tévedett, és nem jelezte, hogy bizonytalan.")

### Feladat 9.2 – a te szavaiddal

Miért nem szabad a modell számára hagyatkozni, és mit csinálsz helyette? Egy-két mondat, idézőjelek közé.

In [ ]:
magyarazat = None  # a None helyére írd a magyarázatod, idézőjelek közé

In [ ]:
ellenoriz_feltetel("magyarazat", lambda s: isinstance(s, str) and len(s.split()) >= 8,
                   "legalább egy teljes mondat (8 szó)")

A számot a Python adja, a modell csak megfogalmazza:

In [ ]:
szam_szovegben = f"{helyes_atlag:.2f}".replace(".", ",")   # magyar tizedesvesszővel

mondat = kerdez(
    "Fogalmazd meg ezt a tényt egy tárgyilagos mondatban közgazdász hallgatóknak, a számot változtatás nélkül átvéve: "
    "2023 januárja és 2024 decembere között a magyar HICP-infláció (éves változás) havi értékeinek átlaga "
    f"{szam_szovegben}% volt. Forrás: Eurostat.",
    rendszer="Egyetlen mondatot írj, bevezető, felsorolás és megjegyzés nélkül.",
    gondolkodas="none",
)
print(mondat)
ellenoriz_feltetel("mondat", lambda m: szam_szovegben in m or szam_szovegben.replace(",", ".") in m,
                   f"a mondatban a Python által számolt {szam_szovegben} szerepel")

## 10. Gyakorlat: saját rendszerprompt

In [ ]:
kerdesek = [
    "Mi a különbség a KSH fogyasztóiár-indexe és az Eurostat HICP-je között?",
    "Mit jelent, ha a reálbér csökken?",
    "Miért emeli a jegybank a kamatot, ha magas az infláció?",
]

### Feladat 10.1 – a rendszerprompt

Írd meg a `sajat_rendszer` változóba. Csak az kerüljön bele, ami **minden** kérdésre igaz.

In [ ]:
sajat_rendszer = None  # a None helyére írd a rendszerpromptod, idézőjelek közé

In [ ]:
ellenoriz_feltetel("sajat_rendszer", lambda s: isinstance(s, str) and len(s) >= 20,
                   "van rendszerprompt (legalább 20 karakter)")

### Feladat 10.2 – Copilottal

> Kérdezd meg a `kerdesek` lista minden kérdését a `kerdez` függvénnyel, a `sajat_rendszer` rendszerprompttal és `gondolkodas="none"` beállítással. Minden kérdés után írd ki a kérdést, a választ, és a válasz hosszát karakterben. A válaszokat gyűjtsd a `valaszok` listába.

Olvasd el: a ciklus minden kérdésnél ugyanazt a rendszerpromptot adja át?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz_feltetel("valaszok", lambda v: len(v) == len(kerdesek) and all(isinstance(x, str) and x.strip() for x in v),
                   "minden kérdésre jött szöveges válasz")

Olvasd el a válaszokat: van bennük tartalmi hiba, pontatlan fogalom, angol szó?